In [1]:
import torch
import torch.nn as nn
import torchvision.models as models
from torch.autograd import Function

class GradientReversal(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        # Save alpha (lambda) for the backward pass
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        # Reverses the gradient by multiplying by -alpha
        output = grad_output.neg() * ctx.alpha
        return output, None

## `adv@conv4` i.e the second last layer

In [2]:
class DebiasedResNetConv4(nn.Module):
    def __init__(self, num_verbs, num_genders=2):
        super(DebiasedResNetConv4, self).__init__()
        
        # Load Pretrained ResNet
        resnet = models.resnet50(pretrained=True)
        children = list(resnet.children())
        
        # --- 1. SPLIT THE BACKBONE ---
        # Part 1: Input -> Conv1 -> ... -> Layer3
        # Output is [Batch, 1024, 14, 14]
        self.features_part1 = nn.Sequential(*children[:7])
        
        # Part 2: Layer4 -> AvgPool
        # Output is [Batch, 2048, 1, 1] -> Flatten -> [Batch, 2048]
        self.features_part2 = nn.Sequential(*children[7:-1])
        
        # Task Classifier
        self.verb_classifier = nn.Linear(resnet.fc.in_features, num_verbs)

        # --- 2. SPATIAL ADVERSARY ---
        # [cite_start]Paper: "3 convolutional layers and 4 linear layers" [cite: 252]
        self.adversary_conv = nn.Sequential(
            # Conv 1
            nn.Conv2d(1024, 512, kernel_size=1), 
            nn.BatchNorm2d(512),
            nn.LeakyReLU(),
            
            # Conv 2
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(),
            
            # Conv 3
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(),
            
            # Global Pooling to flatten
            nn.AdaptiveAvgPool2d((1, 1))
        )
        
        self.adversary_fc = nn.Sequential(
            nn.Flatten(),
            # Linear 1-3
            nn.Linear(256, 256), nn.BatchNorm1d(256), nn.LeakyReLU(),
            nn.Linear(256, 256), nn.BatchNorm1d(256), nn.LeakyReLU(),
            nn.Linear(256, 256), nn.BatchNorm1d(256), nn.LeakyReLU(),
            # Linear 4 (Output)
            nn.Linear(256, num_genders)
        )

    def forward(self, x, alpha=1.0):
        # 1. Extract mid-level features (Conv4)
        f_conv4 = self.features_part1(x)
        
        # 2. Path A: Main Task
        # Finish the ResNet (Layer 4 + AvgPool)
        f_final = self.features_part2(f_conv4)
        f_final = f_final.view(f_final.size(0), -1)
        verb_pred = self.verb_classifier(f_final)
        
        # 3. Path B: Adversary
        # Apply GRL to the spatial features
        reversed_f = GradientReversal.apply(f_conv4, alpha)
        
        # Process with spatial adversary
        adv_feat = self.adversary_conv(reversed_f)
        gender_pred = self.adversary_fc(adv_feat)

        return verb_pred, gender_pred

### Helper Functions

In [3]:
import torch
import json
from torch.utils.data import Dataset
from pathlib import Path

IMSITU_PATH = Path("/content/imsitu_dataset/")

In [4]:
import json

with open(IMSITU_PATH / "imsitu_annotated_train.json", "r") as f:
    train_data = json.load(f)

with open(IMSITU_PATH / "imsitu_annotated_dev.json", "r") as f:
    val_data = json.load(f)

train_verbs = set([item['verb'] for item in train_data])
val_verbs = set([item['verb'] for item in val_data])

all_verbs = train_verbs.union(val_verbs)
verb_to_idx = {v: i for i, v in enumerate(sorted(list(all_verbs)))} 

# verb_to_idx

In [5]:
from PIL import Image

class ImsituDataset(Dataset):
    def __init__(self, source, transform=None, verb2idx=None):
        self.transform = transform

        with open(source, "r") as f:
            self.data = json.load(f)

        # We keep this for reference, but we don't rely on it for model dimension
        self.verbs = sorted(set(i['verb'] for i in self.data))
        self.verb_to_idx = verb2idx

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Load Image
        img_path = IMSITU_PATH / "of500_images_resized" / item['image_path']
        img = Image.open(img_path).convert("RGB")
        
        if self.transform:
            img = self.transform(img)
            
        # FIX 1: Get the verb string first, then map it to the index
        verb_str = item['verb']
        verb = self.verb_to_idx[verb_str] 
        
        # FIX 2: Convert "M"/"F" strings to 0/1 integers for future leakage steps
        # "M" -> 0, "F" -> 1
        gender_str = item['gender']
        gender = 0 if gender_str == "M" else 1

        return img, verb, gender

In [6]:
from torchvision import transforms
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = ImsituDataset(IMSITU_PATH / "imsitu_annotated_train.json", transform=train_transform, verb2idx=verb_to_idx)
val_dataset = ImsituDataset(IMSITU_PATH / "imsitu_annotated_dev.json", transform=val_transform, verb2idx=verb_to_idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Training on {len(train_dataset)} images. Validating on {len(val_dataset)} images.")

Training on 33603 images. Validating on 11211 images.


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm.cli import tqdm  # Specialized for Jupyter
import os

BATCH_SIZE = 32
LR = 1e-4
EPOCHS = 10
SAVE_PATH = "debaised_resnet_adv@conv4.pth"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device {DEVICE}")

Using device cuda


### Training

In [8]:
import numpy as np

def get_alpha(current_step, total_steps):
    """
    Calculates lambda (alpha) using the schedule from Ganin et al. (DANN),
    which is standard for this type of GRL training.
    """
    p = float(current_step) / total_steps
    return 2. / (1. + np.exp(-10 * p)) - 1

In [9]:
num_verbs = len(verb_to_idx)
model = DebiasedResNetConv4(num_verbs=num_verbs).to(DEVICE)

# Optimizers
optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

# Mixed Precision
scaler = torch.cuda.amp.GradScaler()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


/tmp/ipython-input-211318182.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [10]:
best_acc = 0.0
total_steps = EPOCHS * len(train_loader)
current_step = 0

print("Starting Adversarial Training (adv@conv4)...")

for epoch in range(EPOCHS):
    model.train()
    
    total_verb_loss = 0
    total_gender_loss = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)
    
    for images, verbs, genders in pbar:
        images, verbs, genders = images.to(DEVICE), verbs.to(DEVICE), genders.to(DEVICE)
        
        # 1. Calculate Alpha (Lambda) for this step
        alpha = get_alpha(current_step, total_steps)
        current_step += 1
        
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast():
            # 2. Forward Pass
            # We pass alpha so the GRL knows how much to reverse gradients
            verb_pred, gender_pred = model(images, alpha=alpha)
            
            # 3. Calculate Losses
            loss_verb = criterion(verb_pred, verbs)
            loss_gender = criterion(gender_pred, genders)
            
            # 4. Total Loss
            # CRITICAL NOTE: We ADD the losses. 
            # The GRL layer inside the model will automatically FLIP the sign 
            # of the gradient coming from 'gender_pred' during backward().
            # So: Minimize Verb Error AND (Minimize -Gender Error) -> Maximize Gender Error
            total_loss = loss_verb + loss_gender
            
        # 5. Backward & Step
        scaler.scale(total_loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_verb_loss += loss_verb.item()
        total_gender_loss += loss_gender.item()
        
        pbar.set_postfix(v_loss=loss_verb.item(), g_loss=loss_gender.item(), alpha=f"{alpha:.2f}")

    # --- VALIDATION (Check Verb Accuracy) ---
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, verbs, _ in tqdm(val_loader, desc="Validating", leave=False):
            images, verbs = images.to(DEVICE), verbs.to(DEVICE)
            
            # For validation, we don't care about the adversary output or alpha
            verb_pred, _ = model(images, alpha=0.0) 
            
            _, predicted = torch.max(verb_pred.data, 1)
            total += verbs.size(0)
            correct += (predicted == verbs).sum().item()
            
    val_acc = 100 * correct / total
    
    print(f"Epoch {epoch+1} Results:")
    print(f"  Verb Loss: {total_verb_loss/len(train_loader):.4f}")
    print(f"  Gender Loss: {total_gender_loss/len(train_loader):.4f} (Adversary Performance)")
    print(f"  Val Accuracy: {val_acc:.2f}%")
    
    # Save Best
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  --> Best Model Saved")

print("Adversarial Training Complete.")

Starting Adversarial Training (adv@conv4)...


Epoch 1/10:   0%|          | 0/1051 [00:00<?, ?it/s]/tmp/ipython-input-1824844427.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1 Results:
  Verb Loss: 4.7735
  Gender Loss: 0.5639 (Adversary Performance)
  Val Accuracy: 15.03%
  --> Best Model Saved


Epoch 2 Results:
  Verb Loss: 3.7512
  Gender Loss: 0.5761 (Adversary Performance)
  Val Accuracy: 19.26%
  --> Best Model Saved


Epoch 3 Results:
  Verb Loss: 3.2864
  Gender Loss: 0.5798 (Adversary Performance)
  Val Accuracy: 22.73%
  --> Best Model Saved


...

Epoch 6 Results:
  Verb Loss: 2.3759
  Gender Loss: 0.5702 (Adversary Performance)
  Val Accuracy: 26.08%
  --> Best Model Saved


Epoch 7 Results:
  Verb Loss: 2.1343
  Gender Loss: 0.5625 (Adversary Performance)
  Val Accuracy: 24.93%


Epoch 8 Results:
  Verb Loss: 1.9030
  Gender Loss: 0.5535 (Adversary Performance)
  Val Accuracy: 25.97%


Epoch 9 Results:
  Verb Loss: 1.7036
  Gender Loss: 0.5484 (Adversary Performance)
  Val Accuracy: 25.75%


Epoch 10 Results:
  Verb Loss: 1.5120
  Gender Loss: 0.5441 (Adversary Performance)
  Val Accuracy: 26.68%
  --> Best Model Saved
Adversarial Training Complete.


## Calculating Leakage

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from tqdm.notebook import tqdm

class Attacker(nn.Module):
    def __init__(self, input_dim, hidden_dim=300):
        super(Attacker, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(),
            
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(),
            
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(),
            
            nn.Linear(hidden_dim, 2)  # Binary Gender (0 or 1)
        )

    def forward(self, x):
        return self.net(x)

In [12]:
def get_dataset_features(loader, num_verbs, device):
    """
    Extracts Ground Truth features (One-Hot Verbs) and Gender labels.
    """
    all_features = []
    all_genders = []
    
    for _, verbs, genders in tqdm(loader, desc="Extracting Dataset Features", leave=False):
        # Convert integer verbs to One-Hot float tensors
        one_hot = F.one_hot(verbs, num_classes=num_verbs).float()
        
        all_features.append(one_hot)
        all_genders.append(genders)
        
    return torch.cat(all_features).to(device), torch.cat(all_genders).to(device)

def get_model_features(model, loader, device):
    """
    Extracts Model Logits (features) and Gender labels.
    """
    model.eval()
    all_logits = []
    all_genders = []
    
    with torch.no_grad():
        for images, _, genders in tqdm(loader, desc="Extracting Model Logits", leave=False):
            images = images.to(device)
            
            # Forward pass to get logits (before Softmax)
            outputs = model(images)
            logits = outputs[0]
            all_logits.append(logits)
            all_genders.append(genders)
            
    return torch.cat(all_logits).to(device), torch.cat(all_genders).to(device)

In [13]:
from tqdm import tqdm

def train_attacker(X_train, y_train, X_test, y_test, device, name="Attacker"):
    # Create datasets
    train_ds = TensorDataset(X_train, y_train)
    test_ds = TensorDataset(X_test, y_test)
    
    train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
    test_dl = DataLoader(test_ds, batch_size=128, shuffle=False)
    
    # Initialize attacker model
    input_dim = X_train.shape[1]
    model = Attacker(input_dim=input_dim).to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=1e-4)  # Standard LR for MLP
    criterion = nn.CrossEntropyLoss()
    
    best_acc = 0.0
    epochs = 20  # MLPs converge quickly
    
    # ---- Epoch loop with tqdm ----
    for epoch in tqdm(range(epochs), desc=f"{name} Training"):
        model.train()

        # ---- Batch loop with tqdm ----
        train_iter = tqdm(train_dl, desc=f"Epoch {epoch+1}", leave=False)

        for features, targets in train_iter:
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
        
        # ---- Evaluation ----
        model.eval()
        correct = 0
        total = 0
        
        with torch.no_grad():
            for features, targets in test_dl:
                outputs = model(features)
                _, predicted = torch.max(outputs.data, 1)
                total += targets.size(0)
                correct += (predicted == targets).sum().item()
        
        acc = 100 * correct / total
        if acc > best_acc:
            best_acc = acc
    
    print(f"{name} Results | Best Accuracy: {best_acc:.2f}%")
    return best_acc

In [14]:
num_verbs = len(verb_to_idx)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. CALCULATE DATASET LEAKAGE (Lambda_D) ---
print("\n--- Calculating Dataset Leakage (Lambda_D) ---")
# Extract One-Hot Vectors
d_X_train, d_y_train = get_dataset_features(train_loader, num_verbs, DEVICE)
d_X_val, d_y_val = get_dataset_features(val_loader, num_verbs, DEVICE)

# Train Attacker on Ground Truth
lambda_d = train_attacker(d_X_train, d_y_train, d_X_val, d_y_val, DEVICE, name="Dataset Leakage")


--- Calculating Dataset Leakage (Lambda_D) ---


Dataset Leakage Training: 100%|██████████| 20/20 [00:19<00:00,  1.01it/s]       

Dataset Leakage Results | Best Accuracy: 68.09%


In [15]:
num_verbs = len(verb_to_idx)

# A. Instantiate the empty architecture
loaded_model = DebiasedResNetConv4(num_verbs=num_verbs).to(DEVICE)

# B. Load the weights
loaded_model.load_state_dict(torch.load("debaised_resnet_adv@conv4.pth", map_location=DEVICE))

# C. Set to Evaluation Mode (Critical for consistent feature extraction)
loaded_model.eval()

DebiasedResNetConv4(
  (features_part1): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
       

In [16]:
print("\n--- Calculating Model Leakage (Lambda_M) ---")
# Extract Model Logits
m_X_train, m_y_train = get_model_features(loaded_model, train_loader, DEVICE)
m_X_val, m_y_val = get_model_features(loaded_model, val_loader, DEVICE)

# Train Attacker on Model Predictions
lambda_m = train_attacker(m_X_train, m_y_train, m_X_val, m_y_val, DEVICE, name="Model Leakage")


--- Calculating Model Leakage (Lambda_M) ---


Model Leakage Training: 100%|██████████| 20/20 [00:20<00:00,  1.01s/it]     

Model Leakage Results | Best Accuracy: 67.58%


In [17]:
amplification = lambda_m - lambda_d

print("\n" + "="*40)
print(f"Dataset Leakage (Lambda_D): {lambda_d:.2f}%")
print(f"Model Leakage   (Lambda_M): {lambda_m:.2f}%")
print(f"Bias Amplification (Delta): {amplification:.2f}%")
print("="*40)


Dataset Leakage (Lambda_D): 68.09%
Model Leakage   (Lambda_M): 67.58%
Bias Amplification (Delta): -0.52%
